# 06｜決策樹與SVM補充

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/06_決策樹與SVM補充.md`，目前 45 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

樹決策區域、迴歸樹節點、旋轉與穩定性、手刻 CART、SVM margin/hinge/kernel。

保留教材原圖；同概念重畫不保證原書分數相同。架構圖以機制或形狀對照，完整模型案例的範圍在各格說明。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 06-03 | 4、5、7、8、12 | 1. 一棵可閱讀的 Iris 決策樹 |
| 06-04 | 9、10、11 | 2. Gini 與 Entropy，以及投影片手算驗算 |
| 06-05 | 14、15、18 | 3. 手算一次分割：不能只比較左右平均 |
| 06-06 | 3、20、21、22 | 4. 正則化：彎月資料上的邊界 |
| 06-07 | 23、25、26、28 | 5. 迴歸樹：葉節點輸出平均，因為平方損失 |
| 06-08 | 6、24、27、45 | 分類區域、迴歸樹與正則化圖 |
| 06-09 | 30、31、32 | 6. 座標方向與不穩定性 |
| 06-10 | 30、31、32、45 | 旋轉與資料擾動後的決策區域 |
| 06-11 | 33、34、38、39、41 | 7. 補充 SVM：軟間隔 C 與 RBF gamma |
| 06-12 | 40、42、43、44 | 8. 同一切分，三種模型的 CV 比較 |
| 06-13 | 14、15、16、17、18、19、45 | 小型遞迴 CART 與 80 筆計數驗算 |
| 06-14 | 33、34、35、36、37、38、39、41、45 | SVM 間隔、次梯度與核展開 |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 06-01｜投影片 06：2 頁
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4', 'sklearn': 'scikit-learn>=1.4,<2'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 06-02｜投影片 06：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

## 1. 一棵可閱讀的 Iris 決策樹

`export_text` 印出從根到葉的規則；葉節點的 `predict_proba` 來自到達該葉的訓練樣本比例。

In [ ]:
# 程式 06-03｜投影片 06：4、5、7、8、12 頁
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
iris = load_iris()
Xtr, Xte, ytr, yte = train_test_split(iris.data, iris.target, test_size=.3, stratify=iris.target, random_state=SEED)
tree = DecisionTreeClassifier(max_depth=2, random_state=SEED).fit(Xtr, ytr)
print(export_text(tree, feature_names=list(iris.feature_names)))
print('leaf probabilities for first 2 test flowers:')
print(np.round(tree.predict_proba(Xte[:2]), 3))
fig, ax = plt.subplots(figsize=(9, 5))
plot_tree(tree, feature_names=iris.feature_names, class_names=list(iris.target_names), filled=True, ax=ax, fontsize=8)
savefig('11_iris_tree')

## 2. Gini 與 Entropy，以及投影片手算驗算

$\text{Gini}=1-\sum_k p_k^2$；$\text{Entropy}=-\sum_k p_k\log_2 p_k$。對應投影片活動「哪個節點比較純？」節點標籤數 `[4,0]`、`[2,2]`、`[3,1]`。

In [ ]:
# 程式 06-04｜投影片 06：9、10、11 頁
def gini(counts):
    p = np.asarray(counts, float); p = p / p.sum()
    return float(1 - np.sum(p ** 2))
def entropy(counts):
    p = np.asarray(counts, float); p = p / p.sum()
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))

nodes = {'[4,0]': [4, 0], '[2,2]': [2, 2], '[3,1]': [3, 1]}
tbl = pd.DataFrame({'Gini': {k: gini(v) for k, v in nodes.items()},
                    'Entropy': {k: entropy(v) for k, v in nodes.items()}})
assert np.allclose(tbl.Gini, [0, .5, .375])
assert np.allclose(tbl.Entropy, [0, 1, 0.81127812])
tbl

## 3. 手算一次分割：不能只比較左右平均

對應投影片活動「算一次分割」。父節點 `[4,4]`（8 筆），方案 A：左 `[3,0]`、右 `[1,4]`；方案 B：左 `[2,2]`、右 `[2,2]`。權重是樣本數比例。

In [ ]:
# 程式 06-05｜投影片 06：14、15、18 頁
def weighted_gini(left, right):
    nl, nr = sum(left), sum(right); n = nl + nr
    return nl / n * gini(left) + nr / n * gini(right)
parent = gini([4, 4])
A = weighted_gini([3, 0], [1, 4]); B = weighted_gini([2, 2], [2, 2])
assert np.isclose(parent, .5) and np.isclose(A, .2) and np.isclose(B, .5)
assert np.isclose(parent - A, .3) and np.isclose(parent - B, 0.0)
pd.Series({'parent_Gini': parent, 'planA_weighted_Gini': A, 'planA_drop': parent - A,
           'planB_weighted_Gini': B, 'planB_drop': parent - B})

## 4. 正則化：彎月資料上的邊界

不加限制的樹會把訓練點框得很細。`max_depth`、`min_samples_leaf` 等把有效複雜度壓低。

In [ ]:
# 程式 06-06｜投影片 06：3、20、21、22 頁
from sklearn.datasets import make_moons
Xm, ym = make_moons(n_samples=400, noise=.3, random_state=SEED)
Xmtr, Xmte, ymtr, ymte = train_test_split(Xm, ym, test_size=.3, random_state=SEED)
xx, yy = np.meshgrid(np.linspace(Xm[:, 0].min() - .5, Xm[:, 0].max() + .5, 300),
                     np.linspace(Xm[:, 1].min() - .5, Xm[:, 1].max() + .5, 300))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
rows = []
for ax, kw, name in [(axes[0], {}, 'no limit'), (axes[1], {'min_samples_leaf': 20}, 'min_samples_leaf=20')]:
    t = DecisionTreeClassifier(random_state=SEED, **kw).fit(Xmtr, ymtr)
    zz = t.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=.3, cmap='coolwarm')
    ax.scatter(Xmtr[:, 0], Xmtr[:, 1], c=ymtr, cmap='coolwarm', s=12, edgecolor='k', linewidth=.3)
    ax.set_title(f'{name}\ntrain={t.score(Xmtr, ymtr):.2f} test={t.score(Xmte, ymte):.2f}')
    rows.append({'model': name, 'depth': t.get_depth(), 'train_acc': t.score(Xmtr, ymtr), 'test_acc': t.score(Xmte, ymte)})
savefig('11_tree_regularization')
moons_reg = pd.DataFrame(rows).set_index('model')
assert moons_reg.loc['no limit', 'train_acc'] > moons_reg.loc['min_samples_leaf=20', 'train_acc']
moons_reg

## 5. 迴歸樹：葉節點輸出平均，因為平方損失

對應投影片活動「為什麼預測平均值？」葉內目標 `[1,2,6]`。

In [ ]:
# 程式 06-07｜投影片 06：23、25、26、28 頁
leaf = np.array([1., 2., 6.])
mse_of = {c: float(np.mean((leaf - c) ** 2)) for c in [2, 3, 4]}
mae_of = {c: float(np.mean(np.abs(leaf - c))) for c in [2, 3, 4]}
assert np.isclose(mse_of[3], 14 / 3) and mse_of[3] == min(mse_of.values())   # 平均使 MSE 最小
assert mae_of[2] == min(mae_of.values())                                     # 中位數使 MAE 最小
print('MSE by constant:', {k: round(v, 3) for k, v in mse_of.items()}, '-> best = mean =', leaf.mean())
print('MAE by constant:', {k: round(v, 3) for k, v in mae_of.items()}, '-> best = median =', np.median(leaf))

from sklearn.tree import DecisionTreeRegressor
rng = np.random.default_rng(SEED)
xr = np.sort(rng.uniform(0, 1, 200))[:, None]
yr = np.sin(2 * np.pi * xr[:, 0]) + rng.normal(0, .2, 200)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, d in zip(axes, [2, 5]):
    rt = DecisionTreeRegressor(max_depth=d, random_state=SEED).fit(xr, yr)
    g = np.linspace(0, 1, 500)[:, None]
    ax.scatter(xr, yr, s=8, alpha=.4); ax.plot(g, rt.predict(g), 'r-')
    ax.set_title(f'regression tree, max_depth={d}')
savefig('11_regression_tree')

## 分類區域、迴歸樹與正則化圖

分類圖只用 Iris 的兩個花瓣欄位，對照二維切割。迴歸樹的節點輸出是葉內平均；另固定深度比較 min_samples_leaf。都是訓練幾何示例。

In [ ]:
# 程式 06-08｜投影片 06：6、24、27、45 頁
small_tree=DecisionTreeClassifier(max_depth=2,random_state=SEED).fit(iris.data[:,2:4],iris.target)
gx,gy=np.meshgrid(np.linspace(0,7,160),np.linspace(0,3,120));points=np.c_[gx.ravel(),gy.ravel()]
plt.contourf(gx,gy,small_tree.predict(points).reshape(gx.shape),alpha=.3,cmap='viridis')
plt.scatter(iris.data[:,2],iris.data[:,3],c=iris.target,s=15,cmap='viridis');plt.xlabel('Petal length');plt.ylabel('Petal width');savefig('c06_tree_regions')
reg_small=DecisionTreeRegressor(max_depth=2,random_state=SEED).fit(xr,yr)
plt.figure(figsize=(11,4));plot_tree(reg_small,feature_names=['x'],filled=True,fontsize=8);savefig('c06_regression_nodes')
fig,axes=plt.subplots(1,2,figsize=(10,3.5))
for ax,minimum in zip(axes,[1,20]):
    reg=DecisionTreeRegressor(min_samples_leaf=minimum,random_state=SEED).fit(xr,yr)
    ax.scatter(xr[:,0],yr,s=8);ax.plot(g[:,0],reg.predict(g));ax.set(title=f'min_samples_leaf={minimum}',xlabel='x',ylabel='y')
savefig('c06_regression_regularization')

## 6. 座標方向與不穩定性

樹只能做垂直於座標軸的切割。旋轉特徵（例如用 PCA）後，同一份資料可能用更少的分割就分開。

In [ ]:
# 程式 06-09｜投影片 06：30、31、32 頁
from sklearn.decomposition import PCA
rng = np.random.default_rng(SEED)
base = rng.uniform(-1, 1, (300, 2))
angle = np.deg2rad(30); R = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
Xrot = base @ R.T
lab = (base[:, 0] > 0).astype(int)            # 真實邊界在旋轉前的座標軸上
depth_raw = DecisionTreeClassifier(random_state=SEED).fit(Xrot, lab).get_depth()
Xpca = PCA(n_components=2, random_state=SEED).fit_transform(Xrot)
depth_pca = DecisionTreeClassifier(random_state=SEED).fit(Xpca, lab).get_depth()
# 小幅改動訓練資料 -> 規則可能整組改變
d1 = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(Xrot, lab)
d2 = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(Xrot[:-8], lab[:-8])
# PCA 不保證減少樹深度，這裡比較本次資料的實際結果。
print('tree depth on rotated axes:', depth_raw, ' | after PCA rotation:', depth_pca)
print('root feature changed after dropping 8 rows:', d1.tree_.feature[0] != d2.tree_.feature[0]
      or not np.isclose(d1.tree_.threshold[0], d2.tree_.threshold[0]))

## 旋轉與資料擾動後的決策區域

對照圖 5-7～5-9，PCA 不使用標籤，因此不保證與真正類別邊界對齊。比較原座標、旋轉與兩次樹的預測差異。

In [ ]:
# 程式 06-10｜投影片 06：30、31、32、45 頁
fig,axes=plt.subplots(1,3,figsize=(12,3.5))
for ax,F,model,label in [(axes[0],Xrot,d1,'Rotated coordinates'),(axes[1],Xpca,DecisionTreeClassifier(max_depth=3,random_state=SEED).fit(Xpca,lab),'PCA coordinates'),(axes[2],Xrot,d2,'Eight rows removed')]:
    ux,uy=np.meshgrid(np.linspace(F[:,0].min()-.1,F[:,0].max()+.1,160),np.linspace(F[:,1].min()-.1,F[:,1].max()+.1,160))
    ax.contourf(ux,uy,model.predict(np.c_[ux.ravel(),uy.ravel()]).reshape(ux.shape),cmap='coolwarm',alpha=.3)
    ax.scatter(F[:,0],F[:,1],c=lab,s=6,cmap='coolwarm');ax.set(title=label,xlabel='feature 1',ylabel='feature 2')
savefig('c06_rotation_stability')

## 7. 補充 SVM：軟間隔 C 與 RBF gamma

SVM 用「間隔」比較分隔面。軟間隔 `C` 越大越不容忍違規；RBF `gamma` 越大，每個支持向量影響範圍越小、邊界越彎。

In [ ]:
# 程式 06-11｜投影片 06：33、34、38、39、41 頁
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (gamma, C) in zip(axes, [(0.5, 1), (5, 1), (5, 100)]):
    svm = make_pipeline(StandardScaler(), SVC(kernel='rbf', gamma=gamma, C=C)).fit(Xmtr, ymtr)
    zz = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=.3, cmap='coolwarm')
    ax.scatter(Xmtr[:, 0], Xmtr[:, 1], c=ymtr, cmap='coolwarm', s=12, edgecolor='k', linewidth=.3)
    ax.set_title(f'RBF gamma={gamma}, C={C}\ntest acc={svm.score(Xmte, ymte):.2f}')
savefig('11_svm_rbf')

## 8. 同一切分，三種模型的 CV 比較

樹深度 `[2,4,8,None]`、標準化 logistic、標準化 RBF SVM，用同一份 moons 訓練資料做 5 折 CV。

In [ ]:
# 程式 06-12｜投影片 06：40、42、43、44 頁
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
rows = []
for name, est in {'tree_d2': DecisionTreeClassifier(max_depth=2, random_state=SEED),
                  'tree_d4': DecisionTreeClassifier(max_depth=4, random_state=SEED),
                  'tree_d8': DecisionTreeClassifier(max_depth=8, random_state=SEED),
                  'tree_full': DecisionTreeClassifier(random_state=SEED),
                  'logistic_std': make_pipeline(StandardScaler(), LogisticRegression()),
                  'svm_rbf_std': make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1))}.items():
    cv = cross_val_score(est, Xmtr, ymtr, cv=5)
    est.fit(Xmtr, ymtr)
    rows.append({'model': name, 'CV_accuracy': cv.mean(), 'CV_sd': cv.std(ddof=1), 'test_accuracy': est.score(Xmte, ymte)})
compare = pd.DataFrame(rows).set_index('model')
report('trees_svm', {'gini_entropy_check': tbl.to_dict(), 'split_gain': {'A_drop': .3, 'B_drop': 0.0},
       'moons_regularization': moons_reg.to_dict(), 'tree_depth_raw_vs_pca': [int(depth_raw), int(depth_pca)],
       'model_comparison': compare.to_dict()})
compare

## 小型遞迴 CART 與 80 筆計數驗算

以下自編小型 CART 使用 Gini、數值欄位與深度上限；不是完整套件替代品。80 筆示例用相同子節點計數重算兩種準則。

In [ ]:
# 程式 06-13｜投影片 06：14、15、16、17、18、19、45 頁
for name,left,right in [('A',[30,10],[10,30]),('B',[20,40],[20,0])]:
    weighted_e=(sum(left)*entropy(left)+sum(right)*entropy(right))/80
    print(name,'Gini:',weighted_gini(left,right),'Entropy:',weighted_e,'Gain:',1-weighted_e)
def build_cart(F,Y,depth=0):
    counts=np.bincount(Y,minlength=2);node={'prediction':int(counts.argmax()),'counts':counts.tolist()}
    if depth>=2 or len(np.unique(Y))==1:return node
    best=(np.inf,None,None)
    for j in range(F.shape[1]):
        values=np.unique(F[:,j]);thresholds=(values[:-1]+values[1:])/2
        for threshold in thresholds:
            mask=F[:,j]<=threshold
            score=weighted_gini(np.bincount(Y[mask],minlength=2),np.bincount(Y[~mask],minlength=2))
            if score<best[0]:best=(score,j,threshold)
    if best[1] is None:return node
    _,j,threshold=best;mask=F[:,j]<=threshold
    node.update(feature=j,threshold=float(threshold),left=build_cart(F[mask],Y[mask],depth+1),right=build_cart(F[~mask],Y[~mask],depth+1));return node
def cart_predict(node,row):
    if 'feature' not in node:return node['prediction']
    return cart_predict(node['left'] if row[node['feature']]<=node['threshold'] else node['right'],row)
toy_x=np.arange(8.)[:,None];toy_y=np.array([0,0,0,1,0,1,1,1]);tiny=build_cart(toy_x,toy_y)
print(tiny);print('Predictions:',[cart_predict(tiny,row) for row in toy_x])
# 預測只沿一條路徑走；記錄實際比較的節點數。
def prediction_visits(node, row):
    visits = 1
    while 'feature' in node:
        node = node['left'] if row[node['feature']] <= node['threshold'] else node['right']
        visits += 1
    return visits
print('Prediction node visits:', [prediction_visits(tiny, row) for row in toy_x])

## SVM 間隔、次梯度與核展開

使用同一模型的支持向量驗算 decision_function。hinge loss 是 max(0,1-yz)，分對但 yz<1 仍有損失。多項式與 RBF 都以核矩陣計算相似度。

In [ ]:
# 程式 06-14｜投影片 06：33、34、35、36、37、38、39、41、45 頁
margin=np.array([-.5,.5,1.,2.]);hinge=np.maximum(0,1-margin)
assert np.allclose(hinge,[1.5,.5,0,0]);print(pd.DataFrame({'signed_margin':margin,'hinge':hinge}))
sv=SVC(kernel='rbf',gamma=.7,C=1).fit(Xmtr,ymtr)
K=np.exp(-.7*np.sum((Xmte[:8,None,:]-sv.support_vectors_[None,:,:])**2,axis=2))
manual=K@sv.dual_coef_[0]+sv.intercept_[0]
assert np.allclose(manual,sv.decision_function(Xmte[:8]))
linear=SVC(kernel='linear',C=1).fit(Xmtr,ymtr)
print('Linear geometric margin width:',2/np.linalg.norm(linear.coef_[0]))
w_h=np.array([.2,-.3]);sample=np.array([1.,2.]);label=1
subgrad=w_h-label*sample if label*(sample@w_h)<1 else w_h
print('Half-L2 plus one hinge subgradient:',subgrad)
print('Degree-2 kernel:',(Xmtr[:3]@Xmtr[:3].T+1)**2)

## 練習與參考答案

- 「純度」等於「預測正確」嗎？**否，純度只描述該節點的訓練標籤組成，不保證未見樣本正確。**
- 為什麼分割要看加權不純度而非左右平均？**兩子節點樣本數常不同；權重是樣本數比例。**
- 迴歸葉節點若改用絕對誤差，最佳常數是什麼？**中位數。**
- 小幅改動資料就換一套規則，說明什麼？**單棵樹變異大；集成（下一本）用來降低這種變異。**